# Урок 2. Пишем маленькую Terraria по шагам

В этом уроке мы соберем простую игру в стиле Terraria, но пока без интерфейса, инвентаря, сердечек и врагов.

Мы пойдем маленькими шагами:

1. нарисуем мир из блоков;
2. добавим персонажа;
3. научим персонажа ходить;
4. добавим гравитацию и столкновения с блоками;
5. сделаем большой мир и камеру;
6. научимся выбирать блок мышкой;
7. добавим копание блоков.

Главная цель урока - понять ход рассуждений. Поэтому код здесь проще, чем в большой игре проекта.

## 1. Импорт и настройки

В начале любой маленькой игры удобно собрать настройки в одном месте.

Такие настройки часто называют **константами**. Это обычные переменные, но мы пишем их большими буквами, чтобы показать: во время игры они почти не меняются.

Например:

- `WIDTH` и `HEIGHT` - размер игрового поля;
- `TILE` - размер одного блока;
- `MOVE_SPEED` - скорость ходьбы;
- `GRAVITY` - сила, которая тянет персонажа вниз.

Зачем так делать? Чтобы не искать числа по всему коду. Если захочется сделать блоки крупнее или игрока быстрее, мы поменяем одну строку сверху.


In [ ]:
from api import GameObject, BLACK, DIRT, GRASS, OUTLINE, PLAYER, SKY, STONE, WHITE
from backends.tkinter_backend import run

WIDTH = 160
HEIGHT = 120
TILE = 8

AIR_BLOCK = 0
DIRT_BLOCK = 1
STONE_BLOCK = 2

GRAVITY = 1
MOVE_SPEED = 2
JUMP_SPEED = -8
MAX_FALL_SPEED = 8

print("Игровое поле:", WIDTH, "x", HEIGHT)
print("Размер блока:", TILE, "x", TILE)

## 2. Мир из блоков

Terraria-подобная игра начинается с мира из клеток.

Важно: мы не храним в каждой клетке картинку. Мы храним маленькое число:

- `AIR_BLOCK = 0` - пустота;
- `DIRT_BLOCK = 1` - земля;
- `STONE_BLOCK = 2` - камень.

Почему числа, а не слова вроде `"stone"`? Числа проще, быстрее и занимают меньше памяти. А когда нужно рисовать, мы смотрим на число и выбираем цвет.

Мир будет таблицей: список строк, а в каждой строке список блоков.

```text
blocks[y][x]
```

`x` - номер блока слева направо. `y` - номер блока сверху вниз.

Если размер блока `8x8`, то на экране `160x120` помещается `20x15` блоков.


Перед кодом разберем, что будет внутри `TileWorld`.

- `__init__` создает пустую таблицу блоков.
- `generate` заполняет таблицу землей и камнем.
- `get` безопасно получает блок по координатам.
- `set` меняет блок.
- `solid_at_pixel` проверяет, твердый ли блок в точке экрана.
- `draw` рисует только видимую часть мира.

Метод `get` специально считает всё за пределами мира камнем. Так игрок не сможет случайно выйти за границу карты.


In [ ]:
class TileWorld:
    def __init__(self, width_in_tiles, height_in_tiles):
        self.width = width_in_tiles
        self.height = height_in_tiles
        self.blocks = []
        for y in range(self.height):
            row = []
            for x in range(self.width):
                row.append(AIR_BLOCK)
            self.blocks.append(row)
        self.generate()

    def generate(self):
        surface_y = 8
        for x in range(self.width):
            if x % 9 == 0:
                surface_y -= 1
            if x % 13 == 0:
                surface_y += 1
            surface_y = max(5, min(10, surface_y))

            for y in range(self.height):
                if y < surface_y:
                    self.set(x, y, AIR_BLOCK)
                elif y < surface_y + 3:
                    self.set(x, y, DIRT_BLOCK)
                else:
                    self.set(x, y, STONE_BLOCK)

    def get(self, x, y):
        if x < 0 or x >= self.width or y < 0 or y >= self.height:
            return STONE_BLOCK
        return self.blocks[y][x]

    def set(self, x, y, block):
        if 0 <= x < self.width and 0 <= y < self.height:
            self.blocks[y][x] = block

    def solid_at_pixel(self, x, y):
        if y < 0:
            return False
        tile_x = x // TILE
        tile_y = y // TILE
        return self.get(tile_x, tile_y) != AIR_BLOCK

    def draw(self, gfx, camera_x=0, camera_y=0):
        first_x = camera_x // TILE
        first_y = camera_y // TILE
        offset_x = camera_x % TILE
        offset_y = camera_y % TILE
        cols = WIDTH // TILE + 2
        rows = HEIGHT // TILE + 2

        for row in range(rows):
            tile_y = first_y + row
            for col in range(cols):
                tile_x = first_x + col
                block = self.get(tile_x, tile_y)
                if block == AIR_BLOCK:
                    continue

                x = col * TILE - offset_x
                y = row * TILE - offset_y
                if block == DIRT_BLOCK:
                    gfx.rect(x, y, TILE, TILE, DIRT)
                    if self.get(tile_x, tile_y - 1) == AIR_BLOCK:
                        gfx.rect(x, y, TILE, 2, GRASS)
                elif block == STONE_BLOCK:
                    gfx.rect(x, y, TILE, TILE, STONE)
                gfx.rect(x, y, TILE, 1, OUTLINE)

world = TileWorld(WIDTH // TILE, HEIGHT // TILE)
print("В мире", world.width, "на", world.height, "блоков")

## Этап 1. Только мир

Первая версия игры ничего не обновляет. Она только рисует блоки.

Это нормальный первый этап. В играх часто сначала проверяют: “а вообще мир рисуется?”

Обрати внимание на три метода:

- `__init__` создает мир один раз;
- `update` пока пустой, потому что ничего не движется;
- `draw` очищает экран, рисует мир и показывает кадр.

Так мы отделяем логику от рисования, как в первом уроке.


In [ ]:
class Stage1WorldOnly:
    def __init__(self):
        self.world = TileWorld(WIDTH // TILE, HEIGHT // TILE)

    def update(self, keys):
        pass

    def draw(self, gfx):
        gfx.clear(SKY)
        self.world.draw(gfx)
        gfx.present()

In [ ]:
# Запусти ячейку и закрой окно, когда посмотришь результат.
run(Stage1WorldOnly())

## Этап 2. Добавляем персонажа

Теперь нужен объект игрока.

Игрок - это `GameObject`. У него есть координаты `x`, `y`, ширина `w`, высота `h` и цвет.

Мы делаем персонажа размером `8x16`:

- ширина `8` - один блок;
- высота `16` - два блока.

Это похоже на нашу основную Terraria-игру: персонаж занимает примерно два блока по высоте.

Пока персонаж не двигается. Мы просто проверяем, что он правильно рисуется поверх мира.


In [ ]:
class Stage2Player:
    def __init__(self):
        self.world = TileWorld(WIDTH // TILE, HEIGHT // TILE)
        self.player = GameObject(40, 40, 8, 16, PLAYER)

    def update(self, keys):
        pass

    def draw(self, gfx):
        gfx.clear(SKY)
        self.world.draw(gfx)
        self.player.draw(gfx)
        gfx.present()

In [ ]:
run(Stage2Player())

## Этап 3. Движение влево и вправо

Теперь `update(keys)` начинает работать.

У объекта есть скорость:

- `vx` - скорость по горизонтали;
- `vy` - скорость по вертикали.

Если нажато направление влево, `vx` становится отрицательной. Если вправо - положительной. Если ничего не нажато, `vx = 0`.

После этого вызываем `player.move()`: он прибавляет скорость к координатам.

Пока игрок может проходить сквозь блоки. Это нормально: на этом этапе мы учимся только движению.


In [ ]:
class Stage3Walk:
    def __init__(self):
        self.world = TileWorld(WIDTH // TILE, HEIGHT // TILE)
        self.player = GameObject(40, 40, 8, 16, PLAYER)

    def update(self, keys):
        self.player.vx = 0
        if keys.left:
            self.player.vx = -MOVE_SPEED
        if keys.right:
            self.player.vx = MOVE_SPEED

        self.player.move()
        self.player.keep_inside(WIDTH, HEIGHT)

    def draw(self, gfx):
        gfx.clear(SKY)
        self.world.draw(gfx)
        self.player.draw(gfx)
        gfx.present()

In [ ]:
# Управление: стрелки влево/вправо или A/D.
run(Stage3Walk())

## Этап 4. Гравитация и столкновения

Теперь сделаем самое “игровое”: персонаж должен падать, стоять на земле и прыгать.

Идея гравитации простая:

```python
player.vy += GRAVITY
```

Каждый кадр скорость вниз становится чуть больше.

Но если просто двигать игрока вниз, он провалится сквозь землю. Поэтому нужны столкновения.

Мы будем двигать игрока по двум осям отдельно:

1. сначала по `x` и проверяем стены;
2. потом по `y` и проверяем пол/потолок.

Так проще понять, с какой стороны произошло столкновение. Если двигать сразу и по `x`, и по `y`, код быстро становится запутаннее.

Функция `solid_rect(...)` отвечает на вопрос: “прямоугольник игрока касается твердого блока?”


Теперь посмотрим на код столкновений.

В `solid_rect` мы проверяем четыре угла игрока. Для простого прямоугольного персонажа этого достаточно.

В `move_player_with_collisions` есть важный прием: если после движения игрок оказался внутри блока, мы не просто отменяем движение, а аккуратно ставим его ровно рядом с границей блока. Поэтому персонаж не дрожит и не застревает в полу.


In [ ]:
def solid_rect(world, x, y, w, h):
    return (
        world.solid_at_pixel(x, y)
        or world.solid_at_pixel(x + w - 1, y)
        or world.solid_at_pixel(x, y + h - 1)
        or world.solid_at_pixel(x + w - 1, y + h - 1)
    )


def move_player_with_collisions(player, world):
    player.on_ground = False

    player.x += player.vx
    if solid_rect(world, player.x, player.y, player.w, player.h):
        if player.vx > 0:
            right_tile = (player.x + player.w - 1) // TILE
            player.x = right_tile * TILE - player.w
        elif player.vx < 0:
            left_tile = player.x // TILE
            player.x = (left_tile + 1) * TILE

    player.y += player.vy
    if solid_rect(world, player.x, player.y, player.w, player.h):
        if player.vy > 0:
            bottom_tile = (player.y + player.h - 1) // TILE
            player.y = bottom_tile * TILE - player.h
            player.on_ground = True
        elif player.vy < 0:
            top_tile = player.y // TILE
            player.y = (top_tile + 1) * TILE
        player.vy = 0

In [ ]:
class Stage4Physics:
    def __init__(self):
        self.world = TileWorld(WIDTH // TILE, HEIGHT // TILE)
        self.player = GameObject(40, 16, 8, 16, PLAYER)
        self.player.on_ground = False

    def update(self, keys):
        self.player.vx = 0
        if keys.left:
            self.player.vx = -MOVE_SPEED
        if keys.right:
            self.player.vx = MOVE_SPEED
        if keys.up and self.player.on_ground:
            self.player.vy = JUMP_SPEED
            self.player.on_ground = False

        self.player.vy += GRAVITY
        self.player.vy = min(self.player.vy, MAX_FALL_SPEED)
        move_player_with_collisions(self.player, self.world)

    def draw(self, gfx):
        gfx.clear(SKY)
        self.world.draw(gfx)
        self.player.draw(gfx)
        gfx.present()

In [ ]:
# Управление: влево/вправо, прыжок - вверх или Space.
run(Stage4Physics())

## Этап 5. Большой мир и камера

Экран маленький, а мир может быть большим.

Если мир шире экрана, нельзя просто рисовать игрока в координате `x = 300`: он окажется далеко за правым краем окна.

Для этого нужна камера.

`camera_x` означает: сколько пикселей мира осталось слева за экраном.

Когда рисуем объект, мы вычитаем камеру:

```python
screen_x = world_x - camera_x
```

Игрок хранит настоящую координату в мире, а камера решает, какую часть мира сейчас видно.

Функция `clamp(...)` не дает камере уехать дальше краев мира.


В `Stage5Camera` мир уже шириной `80` блоков, то есть `80 * 8 = 640` пикселей.

Экран показывает только `160` пикселей по ширине. Значит, камера должна двигаться за игроком.

При рисовании мира и игрока мы передаем `camera_x`, чтобы всё сместилось влево на нужное количество пикселей.


In [ ]:
def clamp(value, low, high):
    return max(low, min(high, value))


class Stage5Camera:
    def __init__(self):
        self.world = TileWorld(80, HEIGHT // TILE)
        self.player = GameObject(40, 16, 8, 16, PLAYER)
        self.player.on_ground = False
        self.camera_x = 0

    def update(self, keys):
        self.player.vx = 0
        if keys.left:
            self.player.vx = -MOVE_SPEED
        if keys.right:
            self.player.vx = MOVE_SPEED
        if keys.up and self.player.on_ground:
            self.player.vy = JUMP_SPEED
            self.player.on_ground = False

        self.player.vy += GRAVITY
        self.player.vy = min(self.player.vy, MAX_FALL_SPEED)
        move_player_with_collisions(self.player, self.world)

        max_camera_x = self.world.width * TILE - WIDTH
        self.camera_x = clamp(self.player.x - WIDTH // 2, 0, max_camera_x)

    def draw(self, gfx):
        gfx.clear(SKY)
        self.world.draw(gfx, self.camera_x, 0)
        self.player.draw(gfx, self.camera_x, 0)
        gfx.present()

In [ ]:
run(Stage5Camera())

## Этап 6. Выбор блока

Теперь добавим выбранный блок.

На компьютере `keys.pointer_x` и `keys.pointer_y` - это координаты мышки внутри окна.

Но блоки живут в координатах мира. Если камера сдвинута, нужно прибавить `camera_x`:

```python
world_mouse_x = keys.pointer_x + camera_x
selected_x = world_mouse_x // TILE
```

Деление на `TILE` переводит пиксели в номер блока.

Например, если мышка на пикселе `45`, а размер блока `8`, то номер блока будет `45 // 8 = 5`.


In [ ]:
class Stage6SelectBlock(Stage5Camera):
    def __init__(self):
        super().__init__()
        self.selected_x = None
        self.selected_y = None

    def update(self, keys):
        super().update(keys)
        if keys.pointer_active:
            self.selected_x = (keys.pointer_x + self.camera_x) // TILE
            self.selected_y = keys.pointer_y // TILE

    def draw(self, gfx):
        gfx.clear(SKY)
        self.world.draw(gfx, self.camera_x, 0)
        self.player.draw(gfx, self.camera_x, 0)
        self.draw_selected_block(gfx)
        gfx.present()

    def draw_selected_block(self, gfx):
        if self.selected_x is None or self.selected_y is None:
            return
        x = self.selected_x * TILE - self.camera_x
        y = self.selected_y * TILE
        gfx.rect(x, y, TILE, 1, WHITE)
        gfx.rect(x, y + TILE - 1, TILE, 1, WHITE)
        gfx.rect(x, y, 1, TILE, WHITE)
        gfx.rect(x + TILE - 1, y, 1, TILE, WHITE)

In [ ]:
# Подвигай мышкой по окну: блок под курсором будет выделяться.
run(Stage6SelectBlock())

## Этап 7. Копание блоков

Последний шаг этого урока - разрушение выбранного блока.

Самая простая версия копания:

```python
world.set(x, y, AIR_BLOCK)
```

То есть мы не удаляем клетку из таблицы. Мы меняем ее значение на “воздух”.

Если нажата `button_a`, превращаем выбранный блок в пустоту.

На компьютере `button_a` сейчас срабатывает от левой кнопки мыши или клавиши `>`.

В большой игре мы позже улучшим это: земля будет копаться быстрее, камень дольше, а у блока будет полоска прогресса.


In [ ]:
class Stage7DigBlocks(Stage6SelectBlock):
    def update(self, keys):
        super().update(keys)
        if keys.button_a and self.selected_x is not None and self.selected_y is not None:
            self.world.set(self.selected_x, self.selected_y, AIR_BLOCK)


# ЛКМ или > ломает выбранный блок.
run(Stage7DigBlocks())

## Что получилось

Мы собрали основу Terraria-подобной игры:

- мир хранится как таблица блоков;
- каждый блок хранится числом, а рисуется цветом;
- игрок - это `GameObject` размером `8x16`;
- `update(keys)` двигает игрока;
- `vx` отвечает за движение влево/вправо;
- `vy` отвечает за падение и прыжок;
- гравитация меняет `vy`;
- столкновения не дают проходить сквозь блоки;
- камера показывает только часть большого мира;
- мышка выбирает блок;
- `button_a` ломает выбранный блок.

Что можно менять для экспериментов:

- `MOVE_SPEED` - скорость ходьбы;
- `JUMP_SPEED` - силу прыжка;
- `GRAVITY` - скорость падения;
- `TILE` - размер блока;
- цвета блоков в методе `draw`.

В следующих уроках можно улучшать это постепенно: добавить красивые спрайты, разные скорости копания, инвентарь, врагов, здоровье и сохранение мира.
